# 02 - Preprocessamento e Preparação de Dados

## Objetivo
Preparar os dados para modelagem através de:
- Limpeza e remoção de variáveis desnecessárias
- Tratamento de valores nulos
- Encoding de variáveis categóricas (One-Hot)
- Análise de desequilíbrio de classes
- Balanceamento com SMOTE (se necessário)
- Normalização/Standardização
- Divisão treino/teste

## Estrutura:
Este notebook segue um fluxo linear e didático, explicando cada etapa do preprocessamento.

In [1]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Adicionar src/ ao path
sys.path.append(str(Path().resolve().parents[0]))

# Importações padrão
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Módulos do projeto
from src.data.load_data import load_data
from src.preprocessing.encoding import encode_features
from src.preprocessing.balance import check_class_imbalance, apply_smote

# Configurações
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Importações completadas com sucesso!")

✅ Importações completadas com sucesso!


In [2]:
# 1️⃣ CARREGANDO OS DADOS
print("\n1️⃣ CARREGANDO OS DADOS:\n")

df = load_data()
print(f"   ✅ Dataset carregado com sucesso!")
print(f"   Dimensão original: {df.shape}")
print(f"   Colunas: {df.columns.tolist()}")

# Renomear para versão limpa
df_clean = df.copy()


1️⃣ CARREGANDO OS DADOS:

   ✅ Dataset carregado com sucesso!
   Dimensão original: (7267, 21)
   Colunas: ['customerid', 'churn', 'customer_gender', 'customer_seniorcitizen', 'customer_partner', 'customer_dependents', 'customer_tenure', 'phone_phoneservice', 'phone_multiplelines', 'internet_internetservice', 'internet_onlinesecurity', 'internet_onlinebackup', 'internet_deviceprotection', 'internet_techsupport', 'internet_streamingtv', 'internet_streamingmovies', 'account_contract', 'account_paperlessbilling', 'account_paymentmethod', 'account_charges_monthly', 'account_charges_total']


In [3]:
# Informações do dataset
print("📌 Informações do Dataset:")
print(f"   Tipo de dados:")
print(df.dtypes)
print(f"\n   Valores nulos:")
print(df.isnull().sum())
print(f"\n   Colunas: {df.columns.tolist()}")

📌 Informações do Dataset:
   Tipo de dados:
customerid                       str
churn                            str
customer_gender                  str
customer_seniorcitizen         int64
customer_partner                 str
customer_dependents              str
customer_tenure                int64
phone_phoneservice               str
phone_multiplelines              str
internet_internetservice         str
internet_onlinesecurity          str
internet_onlinebackup            str
internet_deviceprotection        str
internet_techsupport             str
internet_streamingtv             str
internet_streamingmovies         str
account_contract                 str
account_paperlessbilling         str
account_paymentmethod            str
account_charges_monthly      float64
account_charges_total        float64
dtype: object

   Valores nulos:
customerid                     0
churn                        224
customer_gender                0
customer_seniorcitizen         0
customer_partn

In [4]:
# Passo 2: Tratar valores nulos
print("\n2️⃣ Tratando valores nulos...")

null_counts = df_clean.isnull().sum()
if null_counts.sum() > 0:
    print(f"\n   Colunas com valores nulos:")
    for col in null_counts[null_counts > 0].index:
        print(f"   - {col}: {null_counts[col]} valores nulos")
    
    # Estratégia: remover linhas com valores nulos na variável alvo (churn)
    if 'churn' in df_clean.columns:
        inicial = len(df_clean)
        df_clean = df_clean.dropna(subset=['churn'])
        removed = inicial - len(df_clean)
        print(f"\n   ✅ Removidas {removed} linhas com churn nulo")
    
    # Para outras colunas numéricas, preencher com média
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col].fillna(df_clean[col].mean(), inplace=True)
            print(f"   ✅ Coluna {col} preenchida com a média")
    
    # Para colunas categóricas, preencher com moda
    categorical_cols = df_clean.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
            print(f"   ✅ Coluna {col} preenchida com a moda")
else:
    print("   ✅ Nenhum valor nulo encontrado!")

print(f"\n   Dimensão final após limpeza: {df_clean.shape}")


2️⃣ Tratando valores nulos...

   Colunas com valores nulos:
   - churn: 224 valores nulos
   - account_charges_total: 11 valores nulos

   ✅ Removidas 224 linhas com churn nulo
   ✅ Coluna account_charges_total preenchida com a média

   Dimensão final após limpeza: (7043, 21)


In [5]:
# 3️⃣ ONE-HOT ENCODING
print("\n3️⃣ ONE-HOT ENCODING DE VARIÁVEIS CATEGÓRICAS:\n")

df_encoded = encode_features(df_clean)

print(f"   ✅ Encoding concluído!")
print(f"   Dimensão após encoding: {df_encoded.shape}")
print(f"   Variáveis criadas: {df_encoded.shape[1] - 1} (features) + 1 (target)")



3️⃣ ONE-HOT ENCODING DE VARIÁVEIS CATEGÓRICAS:

   ✅ Encoding concluído!
   Dimensão após encoding: (7043, 31)
   Variáveis criadas: 30 (features) + 1 (target)


In [6]:
# Verificar tipos de dados
print("\n   Tipos de dados após encoding:")
print(df_encoded.dtypes.value_counts())
print(f"\n   Todas as variáveis estão numéricas? {df_encoded.dtypes.apply(lambda x: x.name).unique()}")


   Tipos de dados após encoding:
bool       26
int64       3
float64     2
Name: count, dtype: int64

   Todas as variáveis estão numéricas? <StringArray>
['int64', 'float64', 'bool']
Length: 3, dtype: str


In [7]:
# 4️⃣ SEPARAÇÃO DE FEATURES E TARGET
print("\n4️⃣ SEPARAÇÃO DE FEATURES (X) E TARGET (y):\n")

# Verificar colunas
print(f"   Colunas em df_encoded: {df_encoded.columns.tolist()}")

# Separar features e target
y = df_encoded['churn'].astype(int)
X = df_encoded.drop(columns=['churn'])

print(f"   Features (X): {X.shape}")
print(f"   Target (y): {y.shape}")
print(f"\n   Distribuição da classe alvo:")
print(y.value_counts())
print(f"\n   Proporção:")
print(y.value_counts(normalize=True).round(3))


4️⃣ SEPARAÇÃO DE FEATURES (X) E TARGET (y):

   Colunas em df_encoded: ['churn', 'customer_seniorcitizen', 'customer_tenure', 'account_charges_monthly', 'account_charges_total', 'customer_gender_Male', 'customer_partner_Yes', 'customer_dependents_Yes', 'phone_phoneservice_Yes', 'phone_multiplelines_No phone service', 'phone_multiplelines_Yes', 'internet_internetservice_Fiber optic', 'internet_internetservice_No', 'internet_onlinesecurity_No internet service', 'internet_onlinesecurity_Yes', 'internet_onlinebackup_No internet service', 'internet_onlinebackup_Yes', 'internet_deviceprotection_No internet service', 'internet_deviceprotection_Yes', 'internet_techsupport_No internet service', 'internet_techsupport_Yes', 'internet_streamingtv_No internet service', 'internet_streamingtv_Yes', 'internet_streamingmovies_No internet service', 'internet_streamingmovies_Yes', 'account_contract_One year', 'account_contract_Two year', 'account_paperlessbilling_Yes', 'account_paymentmethod_Credit card

In [8]:
# 5️⃣ DIVISÃO TREINO/TESTE (80/20)
print("\n5️⃣ DIVISÃO TREINO/TESTE (80/20):\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"   Treino: {X_train.shape[0]} amostras ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"   Teste:  {X_test.shape[0]} amostras ({X_test.shape[0]/len(X)*100:.1f}%)")

print(f"\n   Distribuição do target no treino:")
print(y_train.value_counts())
print(f"\n   Distribuição do target no teste:")
print(y_test.value_counts())



5️⃣ DIVISÃO TREINO/TESTE (80/20):

   Treino: 5634 amostras (80.0%)
   Teste:  1409 amostras (20.0%)

   Distribuição do target no treino:
churn
0    4139
1    1495
Name: count, dtype: int64

   Distribuição do target no teste:
churn
0    1035
1     374
Name: count, dtype: int64


In [9]:
# 6️⃣ STANDARDIZAÇÃO DOS DADOS
print("\n6️⃣ STANDARDIZAÇÃO DOS DADOS:\n")

print("   📌 Nota: A standardização é necessária para modelos baseados em:")
print("       - Distância (KNN, SVM, Regressão Logística)")
print("       - Redes Neurais")
print("\n   ℹ️ Modelos baseados em árvore NÃO requerem standardização.\n")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"   ✅ Standardização concluída!")
print(f"   X_train escalado: {X_train_scaled.shape}")
print(f"   X_test escalado:  {X_test_scaled.shape}")

# Converter de volta para DataFrame para facilitar trabalho posterior
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)



6️⃣ STANDARDIZAÇÃO DOS DADOS:

   📌 Nota: A standardização é necessária para modelos baseados em:
       - Distância (KNN, SVM, Regressão Logística)
       - Redes Neurais

   ℹ️ Modelos baseados em árvore NÃO requerem standardização.

   ✅ Standardização concluída!
   X_train escalado: (5634, 30)
   X_test escalado:  (1409, 30)


In [10]:
# Verificar se há desequilíbrio significativo
print("⚖️ BALANCEAMENTO COM SMOTE:")

min_class_prop = y_train.value_counts(normalize=True).min()

if min_class_prop < 0.3:
    print(f"\n   Classe minoritária: {min_class_prop*100:.1f}%")
    print(f"   ⚠️ Desequilíbrio detectado! Aplicando SMOTE...")
    
    # Aplicar SMOTE
    smote = SMOTE(random_state=42)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
    
    print(f"\n   ✅ SMOTE aplicado com sucesso!")
    print(f"   Antes: {y_train.value_counts().to_dict()}")
    print(f"   Depois: {pd.Series(y_train_balanced).value_counts().to_dict()}")
    print(f"\n   Dimensão após SMOTE: {X_train_balanced.shape}")
else:
    print(f"\n   Classe minoritária: {min_class_prop*100:.1f}%")
    print(f"   ✅ Desequilíbrio aceitável. SMOTE não é necessário.")
    X_train_balanced = X_train
    y_train_balanced = y_train

⚖️ BALANCEAMENTO COM SMOTE:

   Classe minoritária: 26.5%
   ⚠️ Desequilíbrio detectado! Aplicando SMOTE...

   ✅ SMOTE aplicado com sucesso!
   Antes: {0: 4139, 1: 1495}
   Depois: {0: 4139, 1: 4139}

   Dimensão após SMOTE: (8278, 30)


In [11]:
# Salvar dados processados para próximas fases
print("💾 SALVANDO DADOS PROCESSADOS...\n")

import joblib
from pathlib import Path

# Criar pasta se não existir
data_dir = Path('../data/interim')
data_dir.mkdir(parents=True, exist_ok=True)

# Salvar dados
joblib.dump(X_train_scaled, data_dir / 'X_train_scaled.pkl')
joblib.dump(X_test_scaled, data_dir / 'X_test_scaled.pkl')
joblib.dump(y_train_balanced, data_dir / 'y_train.pkl')
joblib.dump(y_test, data_dir / 'y_test.pkl')
joblib.dump(scaler, data_dir / 'scaler.pkl')

print(f"✅ Dados salvos em: {data_dir}")
print(f"   - X_train_scaled.pkl")
print(f"   - X_test_scaled.pkl")
print(f"   - y_train.pkl")
print(f"   - y_test.pkl")
print(f"   - scaler.pkl")

💾 SALVANDO DADOS PROCESSADOS...

✅ Dados salvos em: ../data/interim
   - X_train_scaled.pkl
   - X_test_scaled.pkl
   - y_train.pkl
   - y_test.pkl
   - scaler.pkl


## 📦 SALVAR DADOS PROCESSADOS (Opcional)

In [12]:
# Resumo final
print("="*60)
print("📋 RESUMO DO PREPROCESSAMENTO".center(60))
print("="*60)

print(f"\n📊 DADOS ORIGINAIS:")
print(f"   Dimensão: {df.shape}")

print(f"\n🧹 DEPOIS DA LIMPEZA:")
print(f"   Variáveis removidas: {df.shape[1] - df_clean.shape[1]}")
print(f"   Dimensão: {df_clean.shape}")

print(f"\n🔢 DEPOIS DO ENCODING:")
print(f"   Novos features: {df_encoded.shape[1]}")
print(f"   Dimensão: {df_encoded.shape}")

print(f"\n✂️ SEPARAÇÃO X/Y:")
print(f"   Features (X): {X.shape}")
print(f"   Target (y): {y.shape}")

print(f"\n📊 TREINO/TESTE (80/20):")
print(f"   Treino: {X_train.shape}")
print(f"   Teste: {X_test.shape}")

if min_class_prop < 0.3:
    print(f"\n⚖️ DEPOIS DO SMOTE:")
    print(f"   Treino balanceado: {X_train_balanced.shape}")

print(f"\n📏 DEPOIS DA STANDARDIZAÇÃO:")
print(f"   Treino escalado: {X_train_scaled.shape}")
print(f"   Teste escalado: {X_test_scaled.shape}")

print(f"\n" + "="*60)
print("✅ PREPROCESSAMENTO CONCLUÍDO COM SUCESSO!".center(60))
print("="*60)

                📋 RESUMO DO PREPROCESSAMENTO                

📊 DADOS ORIGINAIS:
   Dimensão: (7267, 21)

🧹 DEPOIS DA LIMPEZA:
   Variáveis removidas: 0
   Dimensão: (7043, 21)

🔢 DEPOIS DO ENCODING:
   Novos features: 31
   Dimensão: (7043, 31)

✂️ SEPARAÇÃO X/Y:
   Features (X): (7043, 30)
   Target (y): (7043,)

📊 TREINO/TESTE (80/20):
   Treino: (5634, 30)
   Teste: (1409, 30)

⚖️ DEPOIS DO SMOTE:
   Treino balanceado: (8278, 30)

📏 DEPOIS DA STANDARDIZAÇÃO:
   Treino escalado: (5634, 30)
   Teste escalado: (1409, 30)

         ✅ PREPROCESSAMENTO CONCLUÍDO COM SUCESSO!          


## 🔟 RESUMO DO PREPROCESSAMENTO

In [ ]:
# Standardização dos dados (média=0, std=1)
print("📏 STANDARDIZAÇÃO DOS DADOS:")
print("\n   📌 Nota: A standardização é necessária para modelos baseados em:")
print("       - Distância (KNN, SVM, Regressão Logística)")
print("       - Redes Neurais")
print("\n   ℹ️ Modelos baseados em árvore NÃO requerem standardização.\n")

scaler = StandardScaler()

# Fit no treino, transform no treino e teste
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

print(f"   ✅ Standardização concluída!")
print(f"\n   Verificação (Treino):")
print(f"   Média: {X_train_scaled.mean(axis=0)[:5]}...")
print(f"   Std: {X_train_scaled.std(axis=0)[:5]}...")

# Converter para DataFrame para melhor visualização
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)

## 9️⃣ NORMALIZAÇÃO/STANDARDIZAÇÃO

## 8️⃣ BALANCEAMENTO COM SMOTE (OPCIONAL)

In [ ]:
# Dividir dados em treino e teste (80/20)
print("📊 DIVISÃO TREINO/TESTE (80/20):")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Mantém a proporção de classes em ambos os conjuntos
)

print(f"\n   ✅ Divisão concluída:")
print(f"   - Treino: {X_train.shape[0]} amostras ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"   - Teste: {X_test.shape[0]} amostras ({X_test.shape[0]/len(X)*100:.1f}%)")

print(f"\n   Distribuição em Treino:")
print(f"   {y_train.value_counts()}")
print(f"\n   Distribuição em Teste:")
print(f"   {y_test.value_counts()}")

## 7️⃣ DIVISÃO TREINO/TESTE

In [ ]:
# Separar X (features) e y (target)
print("✂️ SEPARAÇÃO X e Y:")

# Procurar coluna de churn (pode ser 'churn' ou 'churn_Yes' após encoding)
target_col = None
for col in df_encoded.columns:
    if 'churn' in col.lower():
        target_col = col
        break

if target_col is None:
    raise ValueError("Coluna de churn não encontrada!")

print(f"\n   Coluna alvo: {target_col}")

X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col]

print(f"\n   ✅ Separação concluída:")
print(f"   - X (Features): {X.shape}")
print(f"   - y (Target): {y.shape}")
print(f"\n   Distribuição de y: \n{y.value_counts()}")

## 6️⃣ SEPARAÇÃO DE FEATURES E TARGET

In [ ]:
# Aplicar One-Hot Encoding
print("🔢 ONE-HOT ENCODING:")

df_encoded = encode_features(df_clean)

print(f"\n   ✅ Encoding aplicado!")
print(f"   Dimensões após encoding: {df_encoded.shape}")
print(f"\n   Primeiras linhas:")
df_encoded.head()

## 5️⃣ ENCODING DE VARIÁVEIS CATEGÓRICAS

In [ ]:
# Analisar distribuição da variável alvo
print("⚖️ ANÁLISE DE DESEQUILÍBRIO:")

if 'churn' in df_clean.columns:
    churn_counts = df_clean['churn'].value_counts()
    churn_props = df_clean['churn'].value_counts(normalize=True)
    
    print(f"\n   Distribuição de Churn:")
    print(f"   {churn_counts}")
    print(f"\n   Proporção (%):\n{(churn_props * 100).round(2)}")
    
    # Visualizar
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Gráfico de barras
    churn_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
    axes[0].set_title('Distribuição de Churn (Contagem)', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Quantidade')
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
    
    # Pizza
    axes[1].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%', 
                colors=['#2ecc71', '#e74c3c'], startangle=90)
    axes[1].set_title('Proporção de Churn (%)', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Diagnosticar desequilíbrio
    min_class_prop = churn_props.min()
    if min_class_prop < 0.3:
        print(f"\n   ⚠️  DESEQUILÍBRIO DETECTADO: {min_class_prop*100:.1f}% na classe minoritária")
        print(f"   💡 Sugestão: Aplicar SMOTE para balanceamento")
    else:
        print(f"\n   ✅ Desequilíbrio aceitável (minorities: {min_class_prop*100:.1f}%)")

## 4️⃣ ANÁLISE DE DESEQUILÍBRIO DE CLASSES

In [ ]:
# Passo 1: Remover variáveis desnecessárias (IDs e identificadores únicos)
print("🧹 LIMPEZA DE DADOS:")
print("\n1️⃣ Removendo variáveis desnecessárias...")

# Identificar colunas que podem ser identidades
cols_to_drop = []
for col in df.columns:
    if 'id' in col.lower() or col.lower() in ['customerid']:
        cols_to_drop.append(col)
        print(f"   ❌ Removendo: {col}")

df_clean = df.drop(columns=cols_to_drop, errors='ignore')
print(f"\n   ✅ Dimensão após limpeza: {df_clean.shape}")

## 3️⃣ LIMPEZA DE DADOS

In [ ]:
# Carregar dados do arquivo CSV
df = load_data()

print(f"📊 Dados carregados com sucesso!")
print(f"   Dimensões: {df.shape[0]} linhas × {df.shape[1]} colunas")
print(f"\n📋 Primeiras linhas:")
df.head()

## 2️⃣ CARREGAR E EXPLORAR DADOS BRUTOS

## 1️⃣ IMPORTAÇÕES NECESSÁRIAS

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(5634, 30)
(1409, 30)
(5634,)
(1409,)


In [ ]:
y_train.value_counts()
y_train.value_counts(normalize=True)

churn_Yes
False    0.734647
True     0.265353
Name: proportion, dtype: float64

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)

In [ ]:
X_test_scaled = scaler.transform(X_test)

In [ ]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [ ]:
y_pred = model.predict(X_test_scaled)

In [ ]:
y_pred[:10]

array([False,  True,  True, False, False, False, False, False, False,
        True])

In [ ]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)

print(accuracy)

0.794180269694819
